# 02 -- Tokenizer

Turns raw text into integer ids and back, with no pretrained vocabulary.
Both implementations satisfy the same `Tokenizer` protocol
(`train`/`encode`/`decode`/`save`/`load`/`vocab_size`) so the rest of the
codebase (`qwenmoe/model`, `qwenmoe/data`) never needs to know which one
it's talking to.

*Part of the [training-my-moe code tour](00_overview.ipynb) -- see also: [config](01_config.ipynb) - [tokenizer](02_tokenizer.ipynb) - [data](03_data.ipynb) - [model](04_model.ipynb) - [train](05_train.ipynb) - [viz](06_viz.ipynb).*

In [1]:
from pathlib import Path
import os

# Walk up until we find pyproject.toml, so these cells work no matter where Jupyter's cwd starts.
repo_root = Path.cwd()
while not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
os.chdir(repo_root)
print("Working directory:", Path.cwd())

Working directory: /Users/sjoseph/Documents/Study/training-my-moe


## The `Tokenizer` protocol (`base.py`)
A `@runtime_checkable Protocol` -- any class with the right methods
satisfies it, with no explicit inheritance needed.

In [2]:
from qwenmoe.tokenizer.base import Tokenizer
from qwenmoe.tokenizer.char import CharTokenizer
from qwenmoe.tokenizer.bpe import ByteBPETokenizer

for cls in [CharTokenizer, ByteBPETokenizer]:
    instance = cls()
    print(cls.__name__, "satisfies the Tokenizer protocol:", isinstance(instance, Tokenizer))

CharTokenizer satisfies the Tokenizer protocol: True
ByteBPETokenizer satisfies the Tokenizer protocol: True


## `CharTokenizer` -- vocab = every unique character, sorted
Vocab size is emergent from whatever text it's trained on (not a
configured target) -- used for the tiny-Shakespeare smoke test.

In [3]:
sample_text = "ROMEO:\nBut soft, what light through yonder window breaks?\n"

char_tok = CharTokenizer()
char_tok.train(sample_text)
print("vocab_size:", char_tok.vocab_size)

ids = char_tok.encode("ROMEO:")
print("encode('ROMEO:') ->", ids)
print("decode(...)      ->", repr(char_tok.decode(ids)))
assert char_tok.decode(char_tok.encode(sample_text)) == sample_text
print("round-trip OK")

try:
    char_tok.encode("Z")  # never seen during training
except ValueError as e:
    print("out-of-vocabulary error:", e)

vocab_size: 28
encode('ROMEO:') -> [9, 8, 7, 6, 8, 3]
decode(...)      -> 'ROMEO:'
round-trip OK
out-of-vocabulary error: character 'Z' is not in the trained vocabulary


### save / load round trip

In [4]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "char_tokenizer.json"
    char_tok.save(path)
    reloaded = CharTokenizer()
    reloaded.load(path)
    assert reloaded.vocab_size == char_tok.vocab_size
    assert reloaded.encode(sample_text) == char_tok.encode(sample_text)
    print("save/load round-trip OK, vocab_size:", reloaded.vocab_size)

save/load round-trip OK, vocab_size: 28


## `ByteBPETokenizer` -- from-scratch byte-level BPE
Base vocabulary is the 256 possible byte values, so *any* UTF-8 string
encodes with zero unknown-token cases (unlike `CharTokenizer`). `train`
greedily merges the most frequent adjacent pair, repeatedly, up to
`target_vocab_size`.

In [5]:
bpe_tok = ByteBPETokenizer(target_vocab_size=280)  # 256 base bytes + up to 24 learned merges
bpe_tok.train(sample_text)
print("vocab_size after training:", bpe_tok.vocab_size, "(<= 280)")

ids = bpe_tok.encode("ROMEO:")
print("encode('ROMEO:') ->", ids, f"({len(ids)} tokens for {len('ROMEO:')} characters)")
print("decode(...)      ->", repr(bpe_tok.decode(ids)))
assert bpe_tok.decode(bpe_tok.encode(sample_text)) == sample_text
print("round-trip OK")

# Byte-level -> literally anything round-trips, even text never seen in training.
weird = "h\u00e9llo \U0001F319 world"
assert bpe_tok.decode(bpe_tok.encode(weird)) == weird
print("unseen text (incl. emoji) round-trip OK:", repr(weird))

vocab_size after training: 260 (<= 280)
encode('ROMEO:') -> [82, 79, 77, 69, 79, 58] (6 tokens for 6 characters)
decode(...)      -> 'ROMEO:'
round-trip OK
unseen text (incl. emoji) round-trip OK: 'héllo 🌙 world'


### Compression ratio: char vs. BPE, same text

In [6]:
char_ids = char_tok.encode(sample_text)
bpe_ids = bpe_tok.encode(sample_text)
print(f"characters:      {len(sample_text)}")
print(f"char tokenizer:  {len(char_ids)} tokens (always 1 token/char)")
print(f"BPE tokenizer:   {len(bpe_ids)} tokens ({len(sample_text) / len(bpe_ids):.2f} chars/token)")

characters:      58
char tokenizer:  58 tokens (always 1 token/char)
BPE tokenizer:   49 tokens (1.18 chars/token)


## `build_tokenizer` / `build_tokenizer_from_meta` factories
`build_tokenizer(DataConfig)` picks a tokenizer by `tokenizer_type` when a
full `DataConfig` exists (data prep time). `build_tokenizer_from_meta` does
the same job from just a checkpoint's small `tokenizer_meta` dict, since a
loaded checkpoint doesn't have (or need) a full `DataConfig`.

In [7]:
from qwenmoe.config import DataConfig
from qwenmoe.tokenizer.base import build_tokenizer, build_tokenizer_from_meta

for tt in ["char", "bpe"]:
    built = build_tokenizer(DataConfig(corpus="tiny_shakespeare", tokenizer_type=tt, bpe_vocab_size=300))
    print(tt, "->", type(built).__name__)

# The real tokenizer this repo already trained (tiny-shakespeare, char-level).
real_meta = {"tokenizer_type": "char", "tokenizer_path": ".cache/data/tiny_shakespeare_char/tokenizer.json"}
real_tok = build_tokenizer_from_meta(real_meta)
print("\nreal cached tokenizer vocab_size:", real_tok.vocab_size)
print("encode('ROMEO:') with the REAL trained tokenizer:", real_tok.encode("ROMEO:"))

char -> CharTokenizer
bpe -> ByteBPETokenizer

real cached tokenizer vocab_size: 65
encode('ROMEO:') with the REAL trained tokenizer: [30, 27, 25, 17, 27, 10]
